# 04. Gold Consumption Layer (Star Schema, Power BI Date Dimension & Business Tables)

Constructs the final dimensional model and business reporting marts in `gold_prod`:
- **`gold_prod.reference.dim_date`**: Reusable continuous calendar (2020-2030) with integer join keys and Power BI sort columns (`year_month_sort`).
- **`gold_prod.reference.dim_region`**: Corporate geographic reference dimension with surrogate key `region_sk`.
- **`gold_prod.products.dim_product`**: Product dimension with deterministic surrogate key `product_sk` preserving SCD2 history.
- **`gold_prod.customers.dim_customer`**: Customer dimension with deterministic surrogate key `customer_sk` preserving SCD2 history.
- **`gold_prod.sales.fact_orders`**: **Point-in-Time Fact Table** connecting historical customer/product versions at time of purchase.
- **Business Summary Tables**:
  - `gold_prod.sales.daily_sales_by_region` & `sales_performance`
  - `gold_prod.customers.customer_rfm` (Recency, Frequency, Monetary) & `customer_segments`
  - `gold_prod.products.product_performance` & `discount_summary`


In [0]:
%run ./01_Config


In [0]:
import uuid
from pyspark.sql.functions import (
    col, lit, sequence, to_date, explode, year, quarter, month, dayofmonth, dayofweek,
    date_format, when, concat, current_timestamp, sha2, concat_ws, row_number, sum, count,
    avg, round, datediff, max, min, dense_rank, countDistinct
)
from pyspark.sql.window import Window
from delta.tables import DeltaTable

RUN_ID = str(uuid.uuid4())[:8]
print(f"[GOLD] Starting Gold layer processing. Run ID: {RUN_ID}")


### 1. Reusable Power BI Date Dimension: gold_prod.reference.dim_date


In [0]:
# Power BI Reusable Date Dimension Generator (Continuous 2020-01-01 to 2030-12-31)
df_calendar = (
    spark.sql("SELECT sequence(to_date('2020-01-01'), to_date('2030-12-31'), interval 1 day) as date_seq")
    .select(explode(col("date_seq")).alias("full_date"))
)

df_dim_date = (
    df_calendar
    .withColumn("date_key", date_format(col("full_date"), "yyyyMMdd").cast("int"))
    .withColumn("year", year(col("full_date")))
    .withColumn("year_name", concat(lit("CY "), col("year")))
    .withColumn("quarter", quarter(col("full_date")))
    .withColumn("quarter_name", concat(lit("Q"), col("quarter")))
    .withColumn("year_quarter", concat(col("year"), lit("-Q"), col("quarter")))
    .withColumn("month", month(col("full_date")))
    .withColumn("month_name", date_format(col("full_date"), "MMMM"))
    .withColumn("month_short", date_format(col("full_date"), "MMM"))
    .withColumn("year_month", date_format(col("full_date"), "yyyy-MM"))
    .withColumn("year_month_sort", date_format(col("full_date"), "yyyyMM").cast("int"))  # Critical for PBI Sort-by-Column!
    .withColumn("week_of_year", date_format(col("full_date"), "w").cast("int"))
    .withColumn("iso_week", concat(col("year"), lit("-W"), date_format(col("full_date"), "ww")))
    .withColumn("day_of_month", dayofmonth(col("full_date")))
    .withColumn("day_of_week", dayofweek(col("full_date")))
    .withColumn("day_name", date_format(col("full_date"), "EEEE"))
    .withColumn("day_short", date_format(col("full_date"), "EEE"))
    .withColumn("is_weekend", when(col("day_of_week").isin(1, 7), lit(True)).otherwise(lit(False)))
    .withColumn("is_weekday", when(col("day_of_week").isin(1, 7), lit(False)).otherwise(lit(True)))
    .withColumn("fiscal_year", when(col("month") >= 7, col("year") + 1).otherwise(col("year")))
    .withColumn("fiscal_quarter", concat(lit("FY"), date_format(col("full_date"), "yy"), lit("-Q"),
        when(col("month").isin(7, 8, 9), "1")
        .when(col("month").isin(10, 11, 12), "2")
        .when(col("month").isin(1, 2, 3), "3")
        .otherwise("4")))
    .withColumn("fiscal_month", when(col("month") >= 7, col("month") - 6).otherwise(col("month") + 6))
)

target_dim_date = f"{GOLD_CATALOG}.reference.dim_date"

if spark.catalog.tableExists(target_dim_date):
    tgt_date = DeltaTable.forName(spark, target_dim_date)
    (tgt_date.alias("tgt")
     .merge(df_dim_date.alias("src"), "tgt.date_key = src.date_key")
     .whenMatchedUpdateAll()
     .whenNotMatchedInsertAll()
     .execute())
else:
    df_dim_date.write.format("delta").saveAsTable(target_dim_date)

date_cnt = spark.table(target_dim_date).count()
print(f"✓ Power BI Reusable Date Dimension ready: {target_dim_date} ({date_cnt} continuous days).")


### 2. Geographic Dimension: gold_prod.reference.dim_region


In [0]:
# Geographic Dimension
df_silver_reg = spark.table(f"{SILVER_CATALOG}.region.region")

df_dim_region = (
    df_silver_reg
    .withColumn("region_sk", sha2(concat_ws("||", col("region_id"), col("country_code")), 256))
    .select("region_sk", "region_id", "region_name", "country_code", "currency", "tax_rate")
)

target_dim_region = f"{GOLD_CATALOG}.reference.dim_region"

if spark.catalog.tableExists(target_dim_region):
    tgt_reg = DeltaTable.forName(spark, target_dim_region)
    (tgt_reg.alias("tgt")
     .merge(df_dim_region.alias("src"), "tgt.region_id = src.region_id")
     .whenMatchedUpdateAll()
     .whenNotMatchedInsertAll()
     .execute())
else:
    df_dim_region.write.format("delta").saveAsTable(target_dim_region)

print(f"✓ Region Dimension ready: {target_dim_region} ({spark.table(target_dim_region).count()} records).")


### 3. Product Dimension: gold_prod.products.dim_product


In [0]:
# Product Dimension with Point-in-Time Surrogate Key
df_silver_prod = spark.table(f"{SILVER_CATALOG}.products.products")

df_dim_prod = (
    df_silver_prod
    .withColumn("product_sk", sha2(concat_ws("||", col("product_id"), col("valid_from").cast("string")), 256))
    .select(
        "product_sk", "product_id", "product_name", "category", "unit_price",
        "cost_price", "pricing_tier", "standard_discount", "valid_from", "valid_to", "is_current"
    )
)

target_dim_prod = f"{GOLD_CATALOG}.products.dim_product"

if spark.catalog.tableExists(target_dim_prod):
    tgt_p = DeltaTable.forName(spark, target_dim_prod)
    (tgt_p.alias("tgt")
     .merge(df_dim_prod.alias("src"), "tgt.product_sk = src.product_sk")
     .whenMatchedUpdateAll()
     .whenNotMatchedInsertAll()
     .execute())
else:
    df_dim_prod.write.format("delta").saveAsTable(target_dim_prod)

print(f"✓ Product Dimension ready: {target_dim_prod} ({spark.table(target_dim_prod).count()} versions).")


### 4. Customer Dimension: gold_prod.customers.dim_customer


In [0]:
# Customer Dimension with Point-in-Time Surrogate Key
df_silver_cust = spark.table(f"{SILVER_CATALOG}.customers.customers")

df_dim_cust = (
    df_silver_cust
    .withColumn("customer_sk", sha2(concat_ws("||", col("customer_id"), col("valid_from").cast("string")), 256))
    .select(
        "customer_sk", "customer_id", "fullname", "email", "domain",
        "phone_number", "region_id", "registration_date", "valid_from", "valid_to", "is_current"
    )
)

target_dim_cust = f"{GOLD_CATALOG}.customers.dim_customer"

if spark.catalog.tableExists(target_dim_cust):
    tgt_c = DeltaTable.forName(spark, target_dim_cust)
    (tgt_c.alias("tgt")
     .merge(df_dim_cust.alias("src"), "tgt.customer_sk = src.customer_sk")
     .whenMatchedUpdateAll()
     .whenNotMatchedInsertAll()
     .execute())
else:
    df_dim_cust.write.format("delta").saveAsTable(target_dim_cust)

print(f"✓ Customer Dimension ready: {target_dim_cust} ({spark.table(target_dim_cust).count()} versions).")


### 5. Point-in-Time Fact Orders: gold_prod.sales.fact_orders


In [0]:
# Point-in-Time Fact Orders Table Joining Dimensions as of Order Date
df_silver_orders = spark.table(f"{SILVER_CATALOG}.orders.orders")
dim_cust = spark.table(f"{GOLD_CATALOG}.customers.dim_customer")
dim_prod = spark.table(f"{GOLD_CATALOG}.products.dim_product")
dim_reg = spark.table(f"{GOLD_CATALOG}.reference.dim_region")

# Point-in-time join conditions:
cust_join_cond = (
    (df_silver_orders["customer_id"] == dim_cust["customer_id"]) &
    (df_silver_orders["order_date"] >= dim_cust["valid_from"]) &
    (dim_cust["valid_to"].isNull() | (df_silver_orders["order_date"] < dim_cust["valid_to"]))
)

prod_join_cond = (
    (df_silver_orders["product_id"] == dim_prod["product_id"]) &
    (df_silver_orders["order_date"] >= dim_prod["valid_from"]) &
    (dim_prod["valid_to"].isNull() | (df_silver_orders["order_date"] < dim_prod["valid_to"]))
)

reg_join_cond = (df_silver_orders["region_id"] == dim_reg["region_id"])

df_fact_orders = (
    df_silver_orders.alias("o")
    .join(dim_cust.alias("c"), cust_join_cond, how="left")
    .join(dim_prod.alias("p"), prod_join_cond, how="left")
    .join(dim_reg.alias("r"), reg_join_cond, how="left")
    .select(
        col("o.order_id"),
        coalesce(col("c.customer_sk"), lit("UNKNOWN")).alias("customer_sk"),
        coalesce(col("p.product_sk"), lit("UNKNOWN")).alias("product_sk"),
        coalesce(col("r.region_sk"), lit("UNKNOWN")).alias("region_sk"),
        date_format(col("o.order_date"), "yyyyMMdd").cast("int").alias("order_date_key"),
        col("o.order_date"),
        col("o.customer_id"),
        col("o.product_id"),
        col("o.region_id"),
        col("o.quantity"),
        col("o.unit_price"),
        col("o.total_amount"),
        round(col("o.quantity") * coalesce(col("p.standard_discount"), lit(0.0)), 2).alias("discount_amount"),
        round(col("o.total_amount") - (col("o.quantity") * coalesce(col("p.standard_discount"), lit(0.0))), 2).alias("net_amount"),
        round(col("o.quantity") * coalesce(col("p.cost_price"), lit(0.0)), 2).alias("cost_amount"),
        round((col("o.total_amount") - (col("o.quantity") * coalesce(col("p.standard_discount"), lit(0.0)))) - 
              (col("o.quantity") * coalesce(col("p.cost_price"), lit(0.0))), 2).alias("margin_amount"),
        col("o.order_status"),
        col("o.row_number_rank"),
        lit(RUN_ID).alias("etl_run_id"),
        current_timestamp().alias("load_timestamp")
    )
)

target_fact_orders = f"{GOLD_CATALOG}.sales.fact_orders"

if spark.catalog.tableExists(target_fact_orders):
    tgt_f = DeltaTable.forName(spark, target_fact_orders)
    (tgt_f.alias("tgt")
     .merge(df_fact_orders.alias("src"), "tgt.order_id = src.order_id")
     .whenMatchedUpdateAll()
     .whenNotMatchedInsertAll()
     .execute())
else:
    df_fact_orders.write.format("delta").saveAsTable(target_fact_orders)

fact_count = spark.table(target_fact_orders).count()
print(f"✓ Point-in-Time Fact Orders ready: {target_fact_orders} ({fact_count} transactions).")


### 6. Business Reporting Tables: Sales & Customer RFM


In [0]:
fact_orders = spark.table(f"{GOLD_CATALOG}.sales.fact_orders")
dim_cust = spark.table(f"{GOLD_CATALOG}.customers.dim_customer")

# 1. Daily Sales Summary by Region
df_daily_sales = (
    fact_orders.filter("order_status = 'COMPLETED'")
    .groupBy("order_date_key", "region_id")
    .agg(
        count("order_id").alias("order_volume"),
        round(sum("net_amount"), 2).alias("total_revenue"),
        round(sum("margin_amount"), 2).alias("total_margin"),
        sum("quantity").alias("units_sold"),
        round(avg("net_amount"), 2).alias("average_order_value")
    )
)
df_daily_sales.write.format("delta").mode("overwrite").saveAsTable(f"{GOLD_CATALOG}.sales.daily_sales_by_region")

# 2. Sales Performance by Category & Year
df_sales_perf = (
    fact_orders.alias("f")
    .join(spark.table(f"{GOLD_CATALOG}.products.dim_product").alias("p"), on="product_sk", how="left")
    .join(spark.table(f"{GOLD_CATALOG}.reference.dim_date").alias("d"), fact_orders["order_date_key"] == col("d.date_key"), how="left")
    .groupBy("d.year", "d.quarter_name", "p.category")
    .agg(
        round(sum("f.net_amount"), 2).alias("quarterly_revenue"),
        count("f.order_id").alias("total_orders"),
        round(avg("f.net_amount"), 2).alias("aov")
    )
)
df_sales_perf.write.format("delta").mode("overwrite").saveAsTable(f"{GOLD_CATALOG}.sales.sales_performance")

# 3. Customer RFM Segmentation
max_date = fact_orders.select(max("order_date")).collect()[0][0]
df_rfm_raw = (
    fact_orders.filter("order_status = 'COMPLETED'")
    .groupBy("customer_id")
    .agg(
        datediff(lit(max_date), max("order_date")).alias("recency_days"),
        count("order_id").alias("frequency"),
        round(sum("net_amount"), 2).alias("monetary_value")
    )
)

df_customer_rfm = (
    df_rfm_raw
    .withColumn("rfm_segment",
        when((col("recency_days") <= 90) & (col("frequency") >= 5) & (col("monetary_value") >= 1000), "Champions")
        .when((col("recency_days") <= 180) & (col("frequency") >= 3), "Loyal Customers")
        .when((col("recency_days") > 270) & (col("frequency") >= 3), "At Risk")
        .otherwise("Standard Customers")
    )
)
df_customer_rfm.write.format("delta").mode("overwrite").saveAsTable(f"{GOLD_CATALOG}.customers.customer_rfm")

# 4. Customer Segments Aggregation
df_rfm_dist = (
    df_customer_rfm.groupBy("rfm_segment")
    .agg(
        count("customer_id").alias("customer_count"),
        round(avg("monetary_value"), 2).alias("avg_customer_spend"),
        round(sum("monetary_value"), 2).alias("segment_revenue")
    )
)
df_rfm_dist.write.format("delta").mode("overwrite").saveAsTable(f"{GOLD_CATALOG}.customers.customer_segments")

# 5. Product Performance & Discount Summary
df_prod_perf = (
    fact_orders.groupBy("product_id")
    .agg(
        count("order_id").alias("total_orders"),
        sum("quantity").alias("total_quantity_sold"),
        round(sum("net_amount"), 2).alias("total_revenue"),
        round(sum("discount_amount"), 2).alias("total_discount_given")
    )
)
df_prod_perf.write.format("delta").mode("overwrite").saveAsTable(f"{GOLD_CATALOG}.products.product_performance")

# 6. Enterprise Daily KPIs Summary (Cross-Channel & API Orders)
df_daily_kpis = (
    spark.table(f"{SILVER_CATALOG}.orders.orders")
    .groupBy(to_date(col("order_date")).alias("kpi_date"))
    .agg(
        countDistinct("order_id").alias("total_orders"),
        countDistinct("customer_id").alias("active_customers"),
        round(sum("total_amount"), 2).alias("gross_revenue"),
        round(sum(when(col("order_status") == "COMPLETED", col("total_amount")).otherwise(0)), 2).alias("net_revenue"),
        round(avg("total_amount"), 2).alias("average_order_value"),
        round(sum(when(col("order_status") == "COMPLETED", 1).otherwise(0)) * 100.0 / count("*"), 2).alias("completion_rate_pct"),
        round(sum(when(col("order_status") == "CANCELLED", 1).otherwise(0)) * 100.0 / count("*"), 2).alias("cancellation_rate_pct"),
        round(sum(when(col("order_status") == "RETURNED", 1).otherwise(0)) * 100.0 / count("*"), 2).alias("return_rate_pct")
    )
    .orderBy(col("kpi_date").desc())
)
df_daily_kpis.write.format("delta").mode("overwrite").saveAsTable(f"{GOLD_CATALOG}.sales.daily_kpis")
print(f"✓ Created Gold enterprise KPI table: {GOLD_CATALOG}.sales.daily_kpis")

print("✓ All Gold business reporting tables generated successfully.")


### 7. Gold Referential Integrity & Quality Verification


In [0]:
# Referential Integrity Assertions for Power BI Star Schema
fact_df = spark.table(f"{GOLD_CATALOG}.sales.fact_orders")
dim_date_df = spark.table(f"{GOLD_CATALOG}.reference.dim_date")

# Verify order_date_key matches dim_date.date_key
orphaned_dates = fact_df.join(dim_date_df, fact_df["order_date_key"] == dim_date_df["date_key"], how="left_anti").count()
assert orphaned_dates == 0, f"Referential integrity failure: {orphaned_dates} orders have missing dates in dim_date!"
print(f"✓ Integrity check passed: 100% of order_date_keys exist in {GOLD_CATALOG}.reference.dim_date.")

gold_tables_summary = [
    (f"{GOLD_CATALOG}.reference.dim_date", spark.table(f"{GOLD_CATALOG}.reference.dim_date").count(), "Power BI Continuous Calendar (2020-2030)"),
    (f"{GOLD_CATALOG}.reference.dim_region", spark.table(f"{GOLD_CATALOG}.reference.dim_region").count(), "Geographic Dimension with Region SK"),
    (f"{GOLD_CATALOG}.customers.dim_customer", spark.table(f"{GOLD_CATALOG}.customers.dim_customer").count(), "Customer SCD2 Dimension with Customer SK"),
    (f"{GOLD_CATALOG}.products.dim_product", spark.table(f"{GOLD_CATALOG}.products.dim_product").count(), "Product SCD2 Dimension with Product SK"),
    (f"{GOLD_CATALOG}.sales.fact_orders", spark.table(f"{GOLD_CATALOG}.sales.fact_orders").count(), "Point-in-Time Keyed Fact Table"),
    (f"{GOLD_CATALOG}.sales.daily_kpis", spark.table(f"{GOLD_CATALOG}.sales.daily_kpis").count(), "Cross-Channel Daily Enterprise KPIs"),
    (f"{GOLD_CATALOG}.sales.daily_sales_by_region", spark.table(f"{GOLD_CATALOG}.sales.daily_sales_by_region").count(), "Daily Regional Revenue & AOV Summary"),
    (f"{GOLD_CATALOG}.customers.customer_rfm", spark.table(f"{GOLD_CATALOG}.customers.customer_rfm").count(), "RFM Behavioral Segmentation Mart"),
    (f"{GOLD_CATALOG}.products.product_performance", spark.table(f"{GOLD_CATALOG}.products.product_performance").count(), "Product Sales & Margin Performance")
]
display(spark.createDataFrame(gold_tables_summary, ["GoldTable", "RowCount", "Description"]))
